# MEMM と線形鎖 CRF を実装してラベルバイアス問題を確かめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/memm_vs_crf_scratch.ipynb)

NLP クックブック B-04 ／ 仕組みを理解するための実装

局所正規化を行う最大エントロピーマルコフモデル (MEMM) と、系列全体で大域正規化を行う線形鎖 CRF をまったく同じ素性空間で実装し、遷移の分岐数が偏るデータで MEMM が観測を無視して誤る「ラベルバイアス問題」を数値で確かめたい。

In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# 分岐の非対称性を持つ状態遷移:
# L1 からは L2 の 1 通りしか分岐しないが、M1 からは M2 と M4 の 2 通りに分岐する
STATES = ["L1", "L2", "L3", "M1", "M2", "M3", "M4", "M5"]
S2I = {s: i for i, s in enumerate(STATES)}
K = len(STATES)

train_data = (
    [(["a", "b", "x"], ["L1", "L2", "L3"])] * 40
    + [(["a", "b", "y"], ["M1", "M2", "M3"])] * 18
    + [(["a", "b", "z"], ["M1", "M4", "M5"])] * 18
)

# MEMM と CRF で共通の素性空間: 開始状態、状態遷移、観測の出力
feat_map = {}
def fid(key):
    return feat_map.setdefault(key, len(feat_map))

for s in STATES:
    fid(("START", s))
    for s2 in STATES:
        fid(("TRANS", s, s2))
    for x in ["a", "b", "x", "y", "z"]:
        fid(("EMIT", s, x))
D = len(feat_map)
print("状態数:", K, "/ 素性次元:", D)

状態数: 8 / 素性次元: 112


In [2]:
def memm_nll_grad(w, l2=0.05):
    """MEMM: 各時刻 t で直前ラベル y_{t-1} を固定し、次の候補 y_t だけで局所正規化する"""
    nll, grad = 0.0, np.zeros_like(w)
    for xs, ys in train_data:
        ys_idx = [S2I[y] for y in ys]
        l0 = np.array([w[fid(("START", STATES[y]))] + w[fid(("EMIT", STATES[y], xs[0]))] for y in range(K)])
        z0 = float(logsumexp(l0))
        p0 = np.exp(l0 - z0)
        nll -= l0[ys_idx[0]] - z0
        for y in range(K):
            diff = p0[y] - (1.0 if y == ys_idx[0] else 0.0)
            grad[fid(("START", STATES[y]))] += diff
            grad[fid(("EMIT", STATES[y], xs[0]))] += diff
        for t in range(1, len(xs)):
            yp = ys_idx[t - 1]
            lt = np.array([w[fid(("TRANS", STATES[yp], STATES[y]))] + w[fid(("EMIT", STATES[y], xs[t]))] for y in range(K)])
            zt = float(logsumexp(lt))
            pt = np.exp(lt - zt)
            nll -= lt[ys_idx[t]] - zt
            for y in range(K):
                diff = pt[y] - (1.0 if y == ys_idx[t] else 0.0)
                grad[fid(("TRANS", STATES[yp], STATES[y]))] += diff
                grad[fid(("EMIT", STATES[y], xs[t]))] += diff
    return nll + 0.5 * l2 * np.sum(w ** 2), grad + l2 * w

def crf_nll_grad(w, l2=0.05):
    """線形鎖 CRF: 前向き・後ろ向きアルゴリズムで系列全体の分配関数 Z(x) を求め、大域正規化する"""
    nll, grad = 0.0, np.zeros_like(w)
    trans_mat = np.array([[w[fid(("TRANS", STATES[y1], STATES[y2]))] for y2 in range(K)] for y1 in range(K)])
    start_vec = np.array([w[fid(("START", STATES[y]))] for y in range(K)])
    for xs, ys in train_data:
        T = len(xs)
        ys_idx = [S2I[y] for y in ys]
        emit_mat = np.array([[w[fid(("EMIT", STATES[y], xs[t]))] for y in range(K)] for t in range(T)])

        alpha = np.zeros((T, K))
        alpha[0] = start_vec + emit_mat[0]
        for t in range(1, T):
            alpha[t] = logsumexp(alpha[t - 1][:, None] + trans_mat, axis=0) + emit_mat[t]
        logZ = float(logsumexp(alpha[-1]))

        beta = np.zeros((T, K))
        for t in range(T - 2, -1, -1):
            beta[t] = logsumexp(trans_mat + emit_mat[t + 1][None, :] + beta[t + 1][None, :], axis=1)

        gold = start_vec[ys_idx[0]] + emit_mat[0, ys_idx[0]]
        for t in range(1, T):
            gold += trans_mat[ys_idx[t - 1], ys_idx[t]] + emit_mat[t, ys_idx[t]]
        nll -= gold - logZ

        marg_node = np.exp(alpha + beta - logZ)
        for y in range(K):
            grad[fid(("START", STATES[y]))] += marg_node[0, y] - (1.0 if y == ys_idx[0] else 0.0)
            for t in range(T):
                grad[fid(("EMIT", STATES[y], xs[t]))] += marg_node[t, y] - (1.0 if y == ys_idx[t] else 0.0)
        for t in range(1, T):
            marg_edge = np.exp(alpha[t - 1][:, None] + trans_mat + emit_mat[t][None, :] + beta[t][None, :] - logZ)
            for y1 in range(K):
                for y2 in range(K):
                    grad[fid(("TRANS", STATES[y1], STATES[y2]))] += marg_edge[y1, y2] - (1.0 if (y1 == ys_idx[t - 1] and y2 == ys_idx[t]) else 0.0)
    return nll + 0.5 * l2 * np.sum(w ** 2), grad + l2 * w

w_memm = minimize(memm_nll_grad, np.zeros(D), jac=True, method="L-BFGS-B").x
w_crf = minimize(crf_nll_grad, np.zeros(D), jac=True, method="L-BFGS-B").x

In [3]:
def decode_memm(w, xs):
    T = len(xs)
    dp, bp = np.zeros((T, K)), np.zeros((T, K), dtype=int)
    l0 = np.array([w[fid(("START", STATES[y]))] + w[fid(("EMIT", STATES[y], xs[0]))] for y in range(K)])
    dp[0] = l0 - float(logsumexp(l0))
    for t in range(1, T):
        step = np.zeros((K, K))
        for yp in range(K):
            lt = np.array([w[fid(("TRANS", STATES[yp], STATES[y]))] + w[fid(("EMIT", STATES[y], xs[t]))] for y in range(K)])
            step[yp] = lt - float(logsumexp(lt))
        cand = dp[t - 1][:, None] + step
        bp[t] = cand.argmax(axis=0)
        dp[t] = cand.max(axis=0)
    path = [int(dp[-1].argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(bp[t, path[-1]]))
    return [STATES[i] for i in path[::-1]], float(np.exp(dp[-1].max()))

def decode_crf(w, xs):
    T = len(xs)
    trans_mat = np.array([[w[fid(("TRANS", STATES[y1], STATES[y2]))] for y2 in range(K)] for y1 in range(K)])
    start_vec = np.array([w[fid(("START", STATES[y]))] for y in range(K)])
    emit_mat = np.array([[w[fid(("EMIT", STATES[y], xs[t]))] for y in range(K)] for t in range(T)])
    alpha = np.zeros((T, K))
    alpha[0] = start_vec + emit_mat[0]
    for t in range(1, T):
        alpha[t] = logsumexp(alpha[t - 1][:, None] + trans_mat, axis=0) + emit_mat[t]
    logZ = float(logsumexp(alpha[-1]))
    dp, bp = np.zeros((T, K)), np.zeros((T, K), dtype=int)
    dp[0] = start_vec + emit_mat[0]
    for t in range(1, T):
        cand = dp[t - 1][:, None] + trans_mat + emit_mat[t][None, :]
        bp[t] = cand.argmax(axis=0)
        dp[t] = cand.max(axis=0)
    path = [int(dp[-1].argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(bp[t, path[-1]]))
    return [STATES[i] for i in path[::-1]], float(np.exp(dp[-1].max() - logZ))

print("=== MEMM の局所遷移確率（分岐数の違いによる偏り） ===")
for prev, x in [("L1", "b"), ("M1", "b"), ("L2", "y"), ("M2", "y")]:
    lt = np.array([w_memm[fid(("TRANS", prev, STATES[y]))] + w_memm[fid(("EMIT", STATES[y], x))] for y in range(K)])
    ps = np.exp(lt - float(logsumexp(lt)))
    top = sorted(((round(float(ps[i]), 3), STATES[i]) for i in range(K) if ps[i] > 0.02), reverse=True)
    print(f"  P(y | {prev}, x={x!r}) = {top}")

print("\n=== 復号結果の比較 ===")
tests = [(["a", "b", "x"], ["L1", "L2", "L3"]), (["a", "b", "y"], ["M1", "M2", "M3"]), (["a", "b", "z"], ["M1", "M4", "M5"])]
memm_ok = crf_ok = 0
for xs, gold in tests:
    pm, pm_p = decode_memm(w_memm, xs)
    pc, pc_p = decode_crf(w_crf, xs)
    memm_ok += pm == gold
    crf_ok += pc == gold
    print(f"入力 {xs} / 正解 {gold}")
    print(f"  MEMM: {pm} (P={pm_p:.3f}) {'OK' if pm == gold else 'NG'}")
    print(f"  CRF : {pc} (P={pc_p:.3f}) {'OK' if pc == gold else 'NG'}")
print(f"系列正解率: MEMM {memm_ok}/{len(tests)} vs CRF {crf_ok}/{len(tests)}")

=== MEMM の局所遷移確率（分岐数の違いによる偏り） ===
  P(y | L1, x='b') = [(0.994, 'L2')]
  P(y | M1, x='b') = [(0.496, 'M4'), (0.496, 'M2')]
  P(y | L2, x='y') = [(0.541, 'L3'), (0.381, 'M3')]
  P(y | M2, x='y') = [(0.992, 'M3')]

=== 復号結果の比較 ===
入力 ['a', 'b', 'x'] / 正解 ['L1', 'L2', 'L3']
  MEMM: ['L1', 'L2', 'L3'] (P=0.520) OK
  CRF : ['L1', 'L2', 'L3'] (P=0.985) OK
入力 ['a', 'b', 'y'] / 正解 ['M1', 'M2', 'M3']
  MEMM: ['L1', 'L2', 'L3'] (P=0.282) NG
  CRF : ['M1', 'M2', 'M3'] (P=0.974) OK
入力 ['a', 'b', 'z'] / 正解 ['M1', 'M4', 'M5']
  MEMM: ['L1', 'L2', 'L3'] (P=0.282) NG
  CRF : ['M1', 'M4', 'M5'] (P=0.974) OK
系列正解率: MEMM 1/3 vs CRF 3/3


- MEMM では各時刻で直前の状態ごとに確率の和が 1 になるよう局所正規化するため、遷移先が 1 通りしかない状態 (`L1` → `L2`) には確率約 0.994 が集中し、2 通りに分岐する状態 (`M1` → `M2`/`M4`) はそれぞれ約 0.496 へ割り引かれる。その結果、末尾の観測 `y`/`z` を見ても先頭の優位を覆せず、MEMM はすべて `L1 → L2 → L3` と誤る（ラベルバイアス問題。Lafferty ら 2001）。
- 線形鎖 CRF は系列全体のすべてのラベル列にわたる分配関数 $Z(\mathbf{x})$ で大域正規化するため、末尾の観測 `y`/`z` の証拠が前向き・後ろ向き計算を通じて系列全体に伝わり、3 パターンすべてを高い確信度で正しく復号できる。
- 日本語単語分割で CRF を使う実例は、この付録の「単語分割を文字単位の系列ラベリングとして解く」を参照。